# AVBD 3D Stage 10 — Kaggle CPU + GPU experiment

**Purpose:** fetch the actual `alxspiker/avbd-demo3d` GitHub `main`, compile the C++17 physics engine, run its six test suites, measure up to **1,000,000 actual engine bodies on CPU**, and optionally test an experimental **CUDA translation-only structure-of-arrays kernel**.

**Important:** One million **spatially separated** rigid bodies with **zero contacts** is not equivalent to one million interacting boxes. The CUDA kernel does not include collision detection, rotation, constraints, AVBD dual updates or PCIe transfers. It must not be described as GPU AVBD throughput.

**Kaggle setup:** Notebook Settings → Accelerator → **GPU T4** or **P100** (if testing CUDA). Enable **Internet** to clone GitHub; Kaggle might require account verification. Free CPU-only sessions can run the C++ benchmarks and skip the CUDA experiment. No GitHub Actions or paid APIs.


In [ ]:
import os, pathlib, subprocess, sys, shutil, json, time
def run(*args, cwd=None, check=True):
    print('\n$', ' '.join(map(str,args)), flush=True)
    return subprocess.run(list(map(str,args)),cwd=cwd,check=check)
repo = pathlib.Path('/kaggle/working/avbd-demo3d')
if repo.exists():
    # Prevent accidentally mixing stale sources from a previous notebook run.
    shutil.rmtree(repo)
run('git','clone','--depth','1','--branch','main','https://github.com/alxspiker/avbd-demo3d.git',repo)
run('git','rev-parse','HEAD',cwd=repo)
run('cmake','-S',repo,'-B',repo/'build','-DCMAKE_BUILD_TYPE=Release','-DAVBD_ENABLE_OPENMP=ON')
run('cmake','--build',repo/'build','--parallel','4')
run('ctest','--test-dir',repo/'build','--output-on-failure')


## CPU: run the real World::step engine

`certified` requires the engine to prove non-overlap using conservative predicted AABB occupancy. The baseline (`bvh`) runs the unchanged full AVBD solver. Each prints actual measured wall-clock time. CPU tests are independent of GPU availability.


In [ ]:
for n in [10_000, 100_000, 1_000_000]:
    print('\nCPU SPARSE TEST', n, 'ENGINE BODIES',flush=True)
    run(repo/'build'/'avbd3d_million_benchmark', n, 3, 'certified', cwd=repo)
for n in [10_000, 100_000]:
    print('\nCPU FULL BVH BASELINE',n,flush=True)
    run(repo/'build'/'avbd3d_million_benchmark',n,3,'bvh',cwd=repo)
# Contact-rich workload intentionally remains a separate measurement.
run(repo/'build'/'avbd3d_scaling_benchmark',10000,'islands',3,cwd=repo)


## CPU: isolated SoA layout experiment

The next benchmark integrates only position and velocity in SoA form. It does **not** run collision detection, angular mechanics, AVBD contacts or constraint islands. This isolates data-layout throughput from engine overhead.


In [ ]:
run(repo/'build'/'avbd3d_soa_reference',1_000_000,100,cwd=repo)


## GPU: CUDA SoA prototype (optional, real GPU required)

Uses the actual NVIDIA GPU via CuPy RawKernel/NVRTC and checks sample translations against an independent analytic reference. Device timing excludes host transfers. If the notebook has no CUDA device, it reports **SKIPPED**, not a fictitious speedup. If CuPy is missing, install `cupy-cuda12x` only if your Kaggle CUDA runtime supports CUDA 12.x.


In [ ]:
gpu_ready=False
try:
    import cupy as cp
    gpu_ready = cp.cuda.runtime.getDeviceCount() > 0
except Exception as e:
    print('CuPy/CUDA unavailable:',repr(e))
if gpu_ready:
    run(sys.executable,repo/'tools'/'gpu_freeflight_probe.py','--bodies','1000000','--steps','100',cwd=repo)
else:
    print('GPU EXPERIMENT SKIPPED — not a successful GPU run.')
    print('If an NVIDIA GPU is enabled but CuPy is missing, see the next optional cell.')


### Optional: install CUDA-12 CuPy wheel

Uncomment the installation command below **only if you have a CUDA 12-compatible NVIDIA driver/runtime** and CuPy is missing. Then rerun the preceding GPU cell. The notebook deliberately does not claim automatic GPU fallback.


In [ ]:
# %pip install -q cupy-cuda12x


## Interpretation

- **Million-body CPU success** here means `World::step` advances a million isolated rigid bodies with a certified zero-contact frame, not a dense real-time simulation.
- The SoA CPU/GPU translation kernels are foundational bandwidth experiments, not full physics.
- Compare **same workload, iteration count, precision, and hardware** before claiming CPU-vs-GPU acceleration. The C++ engine uses double precision; the CUDA probe is explicitly float32.
- Include the Git commit hash, GPU type, CPU timings, test results, and validation result in any benchmark report.
